# Forecast Customer ML Trial

Goal:
1. Test whether customer behaviour before launch can predict launch buyers.
2. Calculate real historical buyer ratios for every launch.
3. Use real buyer ratios for forecast calibration.
4. Later train full customer-level ML models for buyer ranking.


In [1]:
import os
import re
import unicodedata
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report
from sklearn.ensemble import HistGradientBoostingClassifier

In [2]:
ORDERS_PATH = "data/raw/orders.csv"
LAUNCHES_PATH = "data/raw/launched_product_details.csv"

print(os.path.exists(ORDERS_PATH), ORDERS_PATH)
print(os.path.exists(LAUNCHES_PATH), LAUNCHES_PATH)

True data/raw/orders.csv
True data/raw/launched_product_details.csv


In [3]:
def normalize_text(x):
    if pd.isna(x):
        return ""
    x = str(x).lower().strip()
    x = unicodedata.normalize("NFKD", x)
    x = "".join([c for c in x if not unicodedata.combining(c)])
    x = re.sub(r"[^a-z0-9äöüß\s]", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x


def clean_numeric(x):
    if pd.isna(x):
        return np.nan

    x = str(x).strip()

    if x in ["", "-", "-%", "nan", "None"]:
        return np.nan

    x = x.replace("€", "")
    x = x.replace("%", "")
    x = x.strip()

    if re.match(r"^\d{1,3}(,\d{3})+$", x):
        x = x.replace(",", "")
    else:
        x = x.replace(",", ".")

    x = re.sub(r"[^0-9.\-]", "", x)

    if x in ["", "-", "."]:
        return np.nan

    return float(x)


def normalize_strategy(x):
    if pd.isna(x):
        return "standard"

    x = normalize_text(x)
    x = x.replace("-", "_").replace(" ", "_")

    mapping = {
        "standard": "standard",
        "standart": "standard",
        "co_creation": "co_creation",
        "cocreation": "co_creation",
        "co": "co_creation",
        "limited_edition": "limited_edition",
        "limited": "limited_edition",
    }

    return mapping.get(x, x)

In [4]:
orders = pd.read_csv(ORDERS_PATH, low_memory=False)
launches = pd.read_csv(LAUNCHES_PATH, sep=",", low_memory=False)

orders.columns = orders.columns.str.strip()
launches.columns = launches.columns.str.strip()

print("orders:", orders.shape)
print("launches:", launches.shape)

display(orders.head())
display(launches.head())

orders: (1580824, 21)
launches: (62, 19)


,order_id,customer_nr,sku,price,date,artikel_name,net_revenue,coupon_code,coupon_product,coupon_channel_short,...,product_category,product,coupon_influencer,quantity,flavour,first_order_date,last_order_date,months_since_first_order,customer_status,nr_of_purchase
0,2000052992,e3de57910d5772b19ea25078e21c146013c257c20e0e0c...,BE20001,37.28972,2021-05-26,MOOD 90 Caps DE,31.696262,imke,NaN,INF,...,Well-being,Mood Kapseln,imkesalander,1,NaN,2021-04-29,2026-04-23,1,RETURNING,2
1,2000077537,4d64fcaa8bf78c45e56073f4e85a0c30de0b9cf3ea2e0d...,BE20001,37.28972,2021-08-24,MOOD 90 Caps DE,31.696262,akhatbrain,NaN,INF,...,Well-being,Mood Kapseln,adrienne_koleszar,1,NaN,2021-08-24,2022-04-22,0,NEW,1
2,2000024030,a64bffe32c15c1697adab6794d31ff6c5f390f0280b273...,BE20001,0.00000,2021-02-22,MOOD 90 Caps DE,0.000000,15be,NaN,INF,...,Well-being,Mood Kapseln,lisa_roeckener,1,NaN,2021-02-22,2021-02-22,0,NEW,1
3,2000108887,8248bcb669a79fed3298f794a0c51cfdabea8256f00bec...,BE20001,37.28972,2021-12-23,MOOD 90 Caps DE,31.696262,colleen,NaN,INF,...,Well-being,Mood Kapseln,colleenschneider_,1,NaN,2021-12-23,2021-12-23,0,NEW,1
4,2000058895,fd04dcc92936f30170c5b1784b272bfd7495fb72f3ac49...,BE20001,37.28972,2021-06-14,MOOD 90 Caps DE,31.696262,wastarasagt,NaN,INF,...,Well-being,Mood Kapseln,wastarasagt,1,NaN,2021-06-14,2021-06-14,0,NEW,1


,sku,artikel_name,product,flavour,product_form,launch_date,first_order_quantity,uvp,launch_strategy_type,Product Use Case / What it is about,Target Group,first_week_quantity_target,first_week_quantity,first_week_quantity_target_accuracy,first_6_week_quantity,first_week_nc,first_6_week_nc,first_week_total_c,first_6_week_total_c
0,BE20351,GUT RESTORE DE/FR/IT 60 caps,Gut Restore,no flavour,Capsules,2023-12-19,"8,000",79.90 €,standart,"Premium 3-in-1 synbiotic (pre-, pro-, and post...",Consumers needing to restore their microbiome ...,2500,1433,57%,4159,611,1943,1352,3919
1,BE20355,DAILY GUT Powder Pomegranate-Hibiscus - 120g P...,Daily Gut Pulver,Pomegranate Hibiscus,Drinking powder,2023-08-07,"1,500",39.90 €,limited_edition,"Supports gut microbiota, hormonal activity, an...",Adults (18+) and especially women seeking impr...,-,1454,-%,2939,558,1034,1096,1949
2,BE20356,DAILY GUT PRO Powder Green Apple - 120g PET DE...,Daily Gut Pulver,Apple,Drinking powder,2023-07-23,"1,500",44.90 €,standart,"Supports gut microbiota, hormonal activity, an...",Adults (18+) and especially women seeking impr...,-,892,-%,1857,266,505,709,1355
3,BE20357,DAILY GUT Powder Peach Ice Tea - 120g PET DE,Daily Gut Pulver,Peach Ice Tea,Drinking powder,2023-08-07,"1,500",39.90 €,limited_edition,"Supports gut microbiota, hormonal activity, an...",Adults (18+) and especially women seeking impr...,-,1453,-%,2914,618,1015,1132,1839
4,BE20358,COLLAGEN + WHEY PULVER Cacao 630g DE/EN,Collagen Whey Pulver,Choco,Drinking powder,2023-08-22,"2,000",49.90 €,standart,2-in-1 protein and beauty drink (21g Whey / 10...,Sporty people and beauty-conscious influencers.,-,71,-%,457,15,102,67,414


In [5]:
orders = orders.copy()
launches = launches.copy()

orders["date"] = pd.to_datetime(orders["date"], errors="coerce")
orders["first_order_date"] = pd.to_datetime(orders["first_order_date"], errors="coerce")
orders["last_order_date"] = pd.to_datetime(orders["last_order_date"], errors="coerce")

orders["quantity"] = pd.to_numeric(orders["quantity"], errors="coerce")
orders["price"] = pd.to_numeric(orders["price"], errors="coerce")
orders["net_revenue"] = pd.to_numeric(orders["net_revenue"], errors="coerce")

orders = orders[
    orders["date"].notna()
    & orders["customer_nr"].notna()
    & orders["sku"].notna()
    & (orders["quantity"].fillna(0) > 0)
].copy()

orders["product_norm"] = orders["product"].apply(normalize_text)
orders["flavour_norm"] = orders["flavour"].apply(normalize_text)
orders["category_norm"] = orders["product_category"].apply(normalize_text)

launches["launch_date"] = pd.to_datetime(launches["launch_date"], errors="coerce")
launches["uvp"] = launches["uvp"].apply(clean_numeric)
launches["first_order_quantity"] = launches["first_order_quantity"].apply(clean_numeric)
launches["launch_strategy_type"] = launches["launch_strategy_type"].apply(normalize_strategy)

for col in [
    "first_week_quantity",
    "first_6_week_quantity",
    "first_week_nc",
    "first_6_week_nc",
    "first_week_total_c",
    "first_6_week_total_c",
]:
    launches[col] = launches[col].apply(clean_numeric)

launches = launches[launches["launch_date"].notna()].copy()

launches["product_norm"] = launches["product"].apply(normalize_text)
launches["flavour_norm"] = launches["flavour"].apply(normalize_text)
launches["product_form_norm"] = launches["product_form"].apply(normalize_text)
launches["launch_month"] = launches["launch_date"].dt.month

print("orders cleaned:", orders.shape)
print("launches cleaned:", launches.shape)

display(launches[[
    "sku", "product", "flavour", "product_form", "launch_date",
    "launch_strategy_type", "first_week_quantity", "first_6_week_quantity",
    "first_week_nc", "first_6_week_nc"
]].head())

orders cleaned: (1580821, 24)
launches cleaned: (62, 23)


,sku,product,flavour,product_form,launch_date,launch_strategy_type,first_week_quantity,first_6_week_quantity,first_week_nc,first_6_week_nc
0,BE20351,Gut Restore,no flavour,Capsules,2023-12-19,standard,1433.0,4159.0,611.0,1943.0
1,BE20355,Daily Gut Pulver,Pomegranate Hibiscus,Drinking powder,2023-08-07,limited_edition,1454.0,2939.0,558.0,1034.0
2,BE20356,Daily Gut Pulver,Apple,Drinking powder,2023-07-23,standard,892.0,1857.0,266.0,505.0
3,BE20357,Daily Gut Pulver,Peach Ice Tea,Drinking powder,2023-08-07,limited_edition,1453.0,2914.0,618.0,1015.0
4,BE20358,Collagen Whey Pulver,Choco,Drinking powder,2023-08-22,standard,71.0,457.0,15.0,102.0


# Part 1 — Single Launch Proof of Concept

This section tests whether customer-level behaviour before one historical launch can predict who bought the launch product within the first week and first six weeks.

In [6]:
launches_sorted = launches.sort_values("launch_date").reset_index(drop=True)

display(launches_sorted[[
    "sku", "product", "flavour", "product_form", "launch_date",
    "launch_strategy_type", "first_week_quantity", "first_6_week_quantity"
]].head(20))


,sku,product,flavour,product_form,launch_date,launch_strategy_type,first_week_quantity,first_6_week_quantity
0,BE20356,Daily Gut Pulver,Apple,Drinking powder,2023-07-23,standard,892.0,1857.0
1,BE20355,Daily Gut Pulver,Pomegranate Hibiscus,Drinking powder,2023-08-07,limited_edition,1454.0,2939.0
2,BE20357,Daily Gut Pulver,Peach Ice Tea,Drinking powder,2023-08-07,limited_edition,1453.0,2914.0
3,BE20358,Collagen Whey Pulver,Choco,Drinking powder,2023-08-22,standard,71.0,457.0
4,BE20359,Collagen Whey Pulver,Vanilla,Drinking powder,2023-08-22,standard,34.0,199.0
5,BE20364,MCT C8/C10,no flavour,Oils,2023-10-16,standard,30.0,219.0
6,BE20365,Sleep Spray,Peppermint,Sprays,2023-10-23,standard,521.0,1728.0
7,BE20371,Daily Gut Pulver,Cinnamon,Drinking powder,2023-10-31,limited_edition,933.0,1880.0
8,BE20351,Gut Restore,no flavour,Capsules,2023-12-19,standard,1433.0,4159.0
9,BE20374,Daily Fiber Drink,Cherry,Drinking powder,2024-02-05,standard,1317.0,3862.0


In [7]:
launch = launches_sorted.iloc[0]

launch_sku = launch["sku"]
launch_date = launch["launch_date"]
launch_product = launch["product"]
launch_flavour = launch["flavour"]

print("Selected launch:")
print("sku:", launch_sku)
print("product:", launch_product)
print("flavour:", launch_flavour)
print("launch_date:", launch_date)


Selected launch:
sku: BE20356
product: Daily Gut Pulver
flavour: Apple
launch_date: 2023-07-23 00:00:00


In [8]:
def build_customer_features_for_launch(orders, launch):
    launch_sku = launch["sku"]
    launch_date = launch["launch_date"]
    launch_product_norm = launch["product_norm"]
    launch_flavour_norm = launch["flavour_norm"]

    before = orders[orders["date"] < launch_date].copy()

    # Only customers who existed before the launch.
    customers = before["customer_nr"].dropna().unique()

    print("Customers before launch:", len(customers))

    # RFM features
    customer_features = (
        before.groupby("customer_nr")
        .agg(
            last_order_date=("date", "max"),
            first_order_date=("date", "min"),
            order_count=("order_id", "nunique"),
            total_quantity=("quantity", "sum"),
            total_revenue=("net_revenue", "sum"),
            avg_price=("price", "mean"),
            avg_units_per_order=("quantity", "mean"),
        )
        .reset_index()
    )

    customer_features["recency_days"] = (
        launch_date - customer_features["last_order_date"]
    ).dt.days

    customer_features["customer_age_days"] = (
        launch_date - customer_features["first_order_date"]
    ).dt.days

    customer_features["avg_order_value"] = (
        customer_features["total_revenue"] / customer_features["order_count"].replace(0, np.nan)
    )

    # Affinity: same product family
    before["is_same_product_family"] = (
        before["product_norm"] == launch_product_norm
    ).astype(int)

    product_aff = (
        before.groupby("customer_nr")["is_same_product_family"]
        .mean()
        .rename("same_product_affinity")
        .reset_index()
    )

    # Affinity: same flavour
    before["is_same_flavour"] = (
        before["flavour_norm"] == launch_flavour_norm
    ).astype(int)

    flavour_aff = (
        before.groupby("customer_nr")["is_same_flavour"]
        .mean()
        .rename("same_flavour_affinity")
        .reset_index()
    )

    # Coupon behavior
    before["used_coupon"] = before["coupon_code"].notna().astype(int)

    coupon_features = (
        before.groupby("customer_nr")
        .agg(
            coupon_usage_share=("used_coupon", "mean"),
            influencer_order_share=("coupon_influencer", lambda s: s.notna().mean()),
        )
        .reset_index()
    )

    # Previous launch-product behavior approximation:
    # customers who bought any SKU that is in launch table before this launch.
    historical_launch_skus = set(
        launches.loc[launches["launch_date"] < launch_date, "sku"].astype(str)
    )

    before["is_previous_launch_sku"] = before["sku"].astype(str).isin(historical_launch_skus).astype(int)

    launch_aff = (
        before.groupby("customer_nr")["is_previous_launch_sku"]
        .agg(["max", "mean"])
        .rename(columns={
            "max": "previous_launch_buyer_flag",
            "mean": "previous_launch_order_share",
        })
        .reset_index()
    )

    # Labels: bought selected launch SKU in first week / first 6 weeks
    first_week_end = launch_date + pd.Timedelta(days=6)
    first_6w_end = launch_date + pd.Timedelta(days=41)

    after_1w = orders[
        (orders["date"] >= launch_date)
        & (orders["date"] <= first_week_end)
        & (orders["sku"].astype(str) == str(launch_sku))
    ]

    after_6w = orders[
        (orders["date"] >= launch_date)
        & (orders["date"] <= first_6w_end)
        & (orders["sku"].astype(str) == str(launch_sku))
    ]

    buyers_1w = set(after_1w["customer_nr"].dropna().astype(str))
    buyers_6w = set(after_6w["customer_nr"].dropna().astype(str))

    customer_features["customer_nr_str"] = customer_features["customer_nr"].astype(str)

    customer_features["bought_1w"] = customer_features["customer_nr_str"].isin(buyers_1w).astype(int)
    customer_features["bought_6w"] = customer_features["customer_nr_str"].isin(buyers_6w).astype(int)

    customer_features = customer_features.drop(columns=["customer_nr_str"])

    # Merge all feature blocks
    df = customer_features.merge(product_aff, on="customer_nr", how="left")
    df = df.merge(flavour_aff, on="customer_nr", how="left")
    df = df.merge(coupon_features, on="customer_nr", how="left")
    df = df.merge(launch_aff, on="customer_nr", how="left")

    # Launch context features
    df["launch_sku"] = launch_sku
    df["launch_month"] = int(launch["launch_month"])
    df["launch_uvp"] = float(launch["uvp"]) if pd.notna(launch["uvp"]) else np.nan
    df["launch_strategy_type"] = launch["launch_strategy_type"]

    # Encode strategy manually for first test
    df["is_co_creation"] = (df["launch_strategy_type"] == "co_creation").astype(int)
    df["is_limited_edition"] = (df["launch_strategy_type"] == "limited_edition").astype(int)

    # Fill missing numeric values
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    df[numeric_cols] = df[numeric_cols].fillna(0)

    return df

In [9]:
one_launch_train = build_customer_features_for_launch(orders, launch)

print(one_launch_train.shape)
display(one_launch_train.head())

print("Positive rate 1w:", one_launch_train["bought_1w"].mean())
print("Positive count 1w:", one_launch_train["bought_1w"].sum())

print("Positive rate 6w:", one_launch_train["bought_6w"].mean())
print("Positive count 6w:", one_launch_train["bought_6w"].sum())

Customers before launch: 140002
(140002, 25)


,customer_nr,last_order_date,first_order_date,order_count,total_quantity,total_revenue,avg_price,avg_units_per_order,recency_days,customer_age_days,...,coupon_usage_share,influencer_order_share,previous_launch_buyer_flag,previous_launch_order_share,launch_sku,launch_month,launch_uvp,launch_strategy_type,is_co_creation,is_limited_edition
0,0000212ede4c061d304dd0c6f7316c4d9638edd6e0529b...,2022-03-01,2021-01-04,2,8,156.892523,23.072430,1.0,509,930,...,1.0,1.0,0,0.0,BE20356,7,44.9,standard,0,0
1,00011f9a3488adfcc9812303eff4f93fda3a57ee5e8b49...,2023-03-12,2023-03-12,1,1,29.355140,32.616822,1.0,133,133,...,1.0,0.0,0,0.0,BE20356,7,44.9,standard,0,0
2,00014eabd6cdd62776ce596434adc3b79e3ecea33f71af...,2023-05-08,2022-05-11,4,4,106.004673,32.616822,1.0,76,438,...,1.0,1.0,0,0.0,BE20356,7,44.9,standard,0,0
3,000179a2312e2aede190aa86b8301ad9c6f33d09bff59e...,2023-01-15,2023-01-15,1,3,22.897196,7.632399,1.0,189,189,...,0.0,0.0,0,0.0,BE20356,7,44.9,standard,0,0
4,0001a83ecbb356f79ee89e048058f63e8f202d64f5df7b...,2021-12-27,2021-04-05,2,3,63.295728,26.770596,1.5,573,839,...,1.0,1.0,0,0.0,BE20356,7,44.9,standard,0,0


Positive rate 1w: 0.003328523878230311
Positive count 1w: 466
Positive rate 6w: 0.00613562663390523
Positive count 6w: 859


In [10]:
feature_cols = [
    "order_count",
    "total_quantity",
    "total_revenue",
    "avg_price",
    "avg_units_per_order",
    "recency_days",
    "customer_age_days",
    "avg_order_value",
    "same_product_affinity",
    "same_flavour_affinity",
    "coupon_usage_share",
    "influencer_order_share",
    "previous_launch_buyer_flag",
    "previous_launch_order_share",
    "launch_month",
    "launch_uvp",
    "is_co_creation",
    "is_limited_edition",
]

X = one_launch_train[feature_cols].fillna(0)
y = one_launch_train["bought_6w"]

print("y positive rate:", y.mean())
print("y positive count:", y.sum())

if y.nunique() < 2:
    print("Cannot train: only one class in target.")
else:
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.25,
        random_state=42,
        stratify=y,
    )

    model = HistGradientBoostingClassifier(
        max_iter=150,
        learning_rate=0.05,
        max_leaf_nodes=31,
        random_state=42,
    )

    model.fit(X_train, y_train)

    p = model.predict_proba(X_test)[:, 1]

    auc = roc_auc_score(y_test, p)
    ap = average_precision_score(y_test, p)

    print("AUC:", auc)
    print("Average precision:", ap)

    pred = (p >= 0.5).astype(int)
    print(classification_report(y_test, pred))

y positive rate: 0.00613562663390523
y positive count: 859
AUC: 0.842267605652635
Average precision: 0.03325956667384074
              precision    recall  f1-score   support

           0       0.99      1.00      1.00     34786
           1       0.00      0.00      0.00       215

    accuracy                           0.99     35001
   macro avg       0.50      0.50      0.50     35001
weighted avg       0.99      0.99      0.99     35001



In [11]:
eval_df = pd.DataFrame({
    "y_true": y_test.values,
    "p_buy": p,
})

overall_rate = eval_df["y_true"].mean()

rows = []
for pct in [0.01, 0.02, 0.05, 0.10, 0.20]:
    n = max(1, int(len(eval_df) * pct))
    top = eval_df.sort_values("p_buy", ascending=False).head(n)

    top_rate = top["y_true"].mean()
    captured_buyers = top["y_true"].sum()
    total_buyers = eval_df["y_true"].sum()

    rows.append({
        "top_percent": pct,
        "n_customers": n,
        "buyer_rate_in_top": top_rate,
        "lift_vs_average": top_rate / overall_rate if overall_rate > 0 else np.nan,
        "captured_buyers": int(captured_buyers),
        "total_buyers": int(total_buyers),
        "recall_in_top": captured_buyers / total_buyers if total_buyers > 0 else np.nan,
    })

topk_df = pd.DataFrame(rows)
display(topk_df)

,top_percent,n_customers,buyer_rate_in_top,lift_vs_average,captured_buyers,total_buyers,recall_in_top
0,0.01,350,0.042857,6.976944,15,215,0.069767
1,0.02,700,0.040000,6.511814,28,215,0.130233
2,0.05,1750,0.037714,6.139710,66,215,0.306977
3,0.10,3500,0.031429,5.116425,110,215,0.511628
4,0.20,7000,0.022857,3.721037,160,215,0.744186


In [12]:
expected_buyers = eval_df["p_buy"].sum()
actual_buyers = eval_df["y_true"].sum()

print("Expected buyers from probability sum:", round(expected_buyers, 1))
print("Actual buyers:", int(actual_buyers))
print("Ratio expected / actual:", round(expected_buyers / actual_buyers, 3))

Expected buyers from probability sum: 211.4
Actual buyers: 215
Ratio expected / actual: 0.983


# Part 2 — Historical Buyer Ratio Table

This section calculates the real buyer penetration ratio for every historical launch:
eligible customers before launch, first-week buyers, first-six-week buyers, buyer ratios, NC shares, and units per customer.

In [13]:
def build_historical_buyer_ratio_table(orders, launches):
    rows = []

    for _, launch in launches.iterrows():
        launch_sku = str(launch["sku"])
        launch_date = launch["launch_date"]

        first_week_end = launch_date + pd.Timedelta(days=6)
        first_6w_end = launch_date + pd.Timedelta(days=41)

        # Customers who existed before this launch
        eligible_customers = set(
            orders.loc[
                orders["date"] < launch_date,
                "customer_nr"
            ].dropna().astype(str)
        )

        eligible_count = len(eligible_customers)

        buyers_1w = set(
            orders.loc[
                (orders["date"] >= launch_date)
                & (orders["date"] <= first_week_end)
                & (orders["sku"].astype(str) == launch_sku),
                "customer_nr"
            ].dropna().astype(str)
        )

        buyers_6w = set(
            orders.loc[
                (orders["date"] >= launch_date)
                & (orders["date"] <= first_6w_end)
                & (orders["sku"].astype(str) == launch_sku),
                "customer_nr"
            ].dropna().astype(str)
        )

        buyers_1w_count = len(buyers_1w)
        buyers_6w_count = len(buyers_6w)

        buyer_ratio_1w = buyers_1w_count / eligible_count if eligible_count > 0 else np.nan
        buyer_ratio_6w = buyers_6w_count / eligible_count if eligible_count > 0 else np.nan

        first_week_total_c = launch.get("first_week_total_c", np.nan)
        first_6_week_total_c = launch.get("first_6_week_total_c", np.nan)
        first_week_nc = launch.get("first_week_nc", np.nan)
        first_6_week_nc = launch.get("first_6_week_nc", np.nan)
        first_week_quantity = launch.get("first_week_quantity", np.nan)
        first_6_week_quantity = launch.get("first_6_week_quantity", np.nan)

        nc_share_1w = (
            first_week_nc / first_week_total_c
            if pd.notna(first_week_nc) and pd.notna(first_week_total_c) and first_week_total_c > 0
            else np.nan
        )

        nc_share_6w = (
            first_6_week_nc / first_6_week_total_c
            if pd.notna(first_6_week_nc) and pd.notna(first_6_week_total_c) and first_6_week_total_c > 0
            else np.nan
        )

        units_per_customer_1w = (
            first_week_quantity / first_week_total_c
            if pd.notna(first_week_quantity) and pd.notna(first_week_total_c) and first_week_total_c > 0
            else np.nan
        )

        units_per_customer_6w = (
            first_6_week_quantity / first_6_week_total_c
            if pd.notna(first_6_week_quantity) and pd.notna(first_6_week_total_c) and first_6_week_total_c > 0
            else np.nan
        )

        rows.append({
            "sku": launch_sku,
            "product": launch.get("product", ""),
            "flavour": launch.get("flavour", ""),
            "product_form": launch.get("product_form", ""),
            "launch_strategy_type": launch.get("launch_strategy_type", ""),
            "launch_date": launch_date,
            "launch_month": launch.get("launch_month", np.nan),
            "uvp": launch.get("uvp", np.nan),

            "eligible_customers_before_launch": eligible_count,

            "buyers_1w_existing": buyers_1w_count,
            "buyers_6w_existing": buyers_6w_count,

            "buyer_ratio_1w_existing": buyer_ratio_1w,
            "buyer_ratio_6w_existing": buyer_ratio_6w,

            "first_week_total_c": first_week_total_c,
            "first_6_week_total_c": first_6_week_total_c,
            "first_week_nc": first_week_nc,
            "first_6_week_nc": first_6_week_nc,

            "nc_share_1w": nc_share_1w,
            "nc_share_6w": nc_share_6w,

            "units_per_customer_1w": units_per_customer_1w,
            "units_per_customer_6w": units_per_customer_6w,

            "first_week_quantity": first_week_quantity,
            "first_6_week_quantity": first_6_week_quantity,
        })

    return pd.DataFrame(rows)

In [14]:
launch_ratio_table = build_historical_buyer_ratio_table(orders, launches)

print("launch_ratio_table:", launch_ratio_table.shape)

display(
    launch_ratio_table[
        [
            "sku",
            "product",
            "flavour",
            "launch_date",
            "launch_strategy_type",
            "eligible_customers_before_launch",
            "buyers_1w_existing",
            "buyers_6w_existing",
            "buyer_ratio_1w_existing",
            "buyer_ratio_6w_existing",
            "first_week_total_c",
            "first_6_week_total_c",
            "first_week_nc",
            "first_6_week_nc",
            "nc_share_1w",
            "nc_share_6w",
            "units_per_customer_1w",
            "units_per_customer_6w",
        ]
    ].sort_values("buyer_ratio_6w_existing", ascending=False).head(20)
)

launch_ratio_table: (62, 23)


,sku,product,flavour,launch_date,launch_strategy_type,eligible_customers_before_launch,buyers_1w_existing,buyers_6w_existing,buyer_ratio_1w_existing,buyer_ratio_6w_existing,first_week_total_c,first_6_week_total_c,first_week_nc,first_6_week_nc,nc_share_1w,nc_share_6w,units_per_customer_1w,units_per_customer_6w
0,BE20351,Gut Restore,no flavour,2023-12-19,standard,159318,10,5647,0.000063,0.035445,1352.0,3919.0,611.0,1943.0,0.451923,0.495790,1.059911,1.061240
13,BE20384,Daily Gut Pulver,Strawberry,2024-04-03,standard,182862,4,6409,0.000022,0.035048,4.0,6409.0,0.0,4004.0,0.000000,0.624746,1.000000,1.288188
59,BE20473,Daily Gut Pulver,Strawberry,2026-03-22,standard,329755,4873,8746,0.014778,0.026523,4873.0,8746.0,2100.0,4458.0,0.430946,0.509719,1.420891,1.367139
14,BE20385,Daily Gut Pulver,Lemon,2024-04-03,standard,182862,5,4358,0.000027,0.023832,5.0,4358.0,0.0,2335.0,0.000000,0.535796,1.000000,1.284305
35,BE20418,Gut Shape,no flavour,2024-12-27,standard,238591,3338,5573,0.013990,0.023358,3338.0,5573.0,1069.0,2028.0,0.320252,0.363897,1.757939,1.884263
16,BE20392,Daily Gut Pulver,Berrymix,2024-04-03,standard,182862,6,3620,0.000033,0.019796,6.0,3620.0,0.0,2097.0,0.000000,0.579282,1.000000,1.222099
10,BE20374,Daily Fiber Drink,Cherry,2024-02-05,standard,169906,1024,3152,0.006027,0.018551,1024.0,3152.0,510.0,1822.0,0.498047,0.578046,1.286133,1.225254
58,BE20472,Daily Gut Pulver,Lemon,2026-03-22,standard,329755,3406,5725,0.010329,0.017361,3406.0,5725.0,1440.0,2666.0,0.422783,0.465677,1.449501,1.386376
12,BE20377,Daily Gut Pulver,Creamy Vanilla,2024-02-19,standard,172656,923,2989,0.005346,0.017312,923.0,2989.0,425.0,1695.0,0.460455,0.567079,1.352113,1.320174
21,BE20400,Daily Gut + Collagen Pulver,Vanilla,2024-06-24,co_creation,204037,2086,3405,0.010224,0.016688,2086.0,3405.0,820.0,1445.0,0.393097,0.424376,1.430010,1.526872


In [15]:
summary_cols = [
    "eligible_customers_before_launch",
    "buyers_1w_existing",
    "buyers_6w_existing",
    "buyer_ratio_1w_existing",
    "buyer_ratio_6w_existing",
    "nc_share_1w",
    "nc_share_6w",
    "units_per_customer_1w",
    "units_per_customer_6w",
]

display(
    launch_ratio_table[summary_cols]
    .describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9])
)

,eligible_customers_before_launch,buyers_1w_existing,buyers_6w_existing,buyer_ratio_1w_existing,buyer_ratio_6w_existing,nc_share_1w,nc_share_6w,units_per_customer_1w,units_per_customer_6w
count,62.000000,62.000000,62.000000,62.000000,62.000000,62.000000,62.000000,62.000000,62.000000
mean,232733.725806,852.209677,2058.080645,0.003615,0.009308,0.331994,0.395254,1.295400,1.317180
std,55843.938338,909.903132,1723.453313,0.003409,0.007915,0.126104,0.102879,0.176386,0.158301
min,140002.000000,4.000000,122.000000,0.000022,0.000799,0.000000,0.218090,1.000000,1.061240
10%,153492.200000,32.600000,397.800000,0.000190,0.001426,0.194632,0.277285,1.098976,1.156037
25%,183783.000000,217.000000,818.500000,0.000776,0.002862,0.276247,0.320610,1.165534,1.206835
50%,234311.000000,663.000000,1617.500000,0.002667,0.007205,0.344072,0.379106,1.284851,1.298420
75%,271302.250000,1124.500000,2790.500000,0.005275,0.013159,0.410228,0.456034,1.411586,1.382402
90%,312813.600000,1738.500000,3809.900000,0.007913,0.018432,0.459602,0.542767,1.520159,1.514557
max,329755.000000,4873.000000,8746.000000,0.014778,0.035445,0.577869,0.641841,1.782443,1.884263


In [16]:
monthly_new_customers = (
    orders.dropna(subset=["first_order_date"])
    .assign(first_order_month=lambda d: d["first_order_date"].dt.to_period("M").astype(str))
    .groupby("first_order_month")["customer_nr"]
    .nunique()
    .reset_index(name="new_customers")
)

monthly_new_customers["date"] = pd.to_datetime(monthly_new_customers["first_order_month"] + "-01")

monthly_new_customers = monthly_new_customers.sort_values("date")

display(monthly_new_customers.tail(24))

print("Recent 3M avg new customers:", monthly_new_customers.tail(3)["new_customers"].mean())
print("Historical avg new customers:", monthly_new_customers["new_customers"].mean())

,first_order_month,new_customers,date
91,2024-05,8195,2024-05-01
92,2024-06,7542,2024-06-01
93,2024-07,7040,2024-07-01
94,2024-08,8145,2024-08-01
95,2024-09,4661,2024-09-01
96,2024-10,5190,2024-10-01
97,2024-11,4192,2024-11-01
98,2024-12,3701,2024-12-01
99,2025-01,6162,2025-01-01
100,2025-02,4527,2025-02-01


Recent 3M avg new customers: 5589.0
Historical avg new customers: 2978.939130434783


In [17]:
launch_ratio_table["launch_year_month"] = (
    launch_ratio_table["launch_date"]
    .dt.to_period("M")
    .astype(str)
)

launch_ratio_table = launch_ratio_table.merge(
    monthly_new_customers[["first_order_month", "new_customers"]],
    left_on="launch_year_month",
    right_on="first_order_month",
    how="left"
)

launch_ratio_table = launch_ratio_table.rename(
    columns={"new_customers": "monthly_new_customers_at_launch"}
)

launch_ratio_table["nc_ratio_1w_vs_monthly_nc"] = (
    launch_ratio_table["first_week_nc"]
    / launch_ratio_table["monthly_new_customers_at_launch"].replace(0, np.nan)
)

launch_ratio_table["nc_ratio_6w_vs_monthly_nc"] = (
    launch_ratio_table["first_6_week_nc"]
    / launch_ratio_table["monthly_new_customers_at_launch"].replace(0, np.nan)
)

display(
    launch_ratio_table[
        [
            "sku",
            "product",
            "launch_date",
            "launch_strategy_type",
            "first_week_nc",
            "first_6_week_nc",
            "monthly_new_customers_at_launch",
            "nc_ratio_1w_vs_monthly_nc",
            "nc_ratio_6w_vs_monthly_nc",
            "nc_share_1w",
            "nc_share_6w",
        ]
    ].sort_values("nc_ratio_6w_vs_monthly_nc", ascending=False).head(20)
)

,sku,product,launch_date,launch_strategy_type,first_week_nc,first_6_week_nc,monthly_new_customers_at_launch,nc_ratio_1w_vs_monthly_nc,nc_ratio_6w_vs_monthly_nc,nc_share_1w,nc_share_6w
0,BE20351,Gut Restore,2023-12-19,standard,611.0,1943.0,3037,0.201185,0.639776,0.451923,0.495790
59,BE20473,Daily Gut Pulver,2026-03-22,standard,2100.0,4458.0,7744,0.271178,0.575671,0.430946,0.509719
35,BE20418,Gut Shape,2024-12-27,standard,1069.0,2028.0,3701,0.288841,0.547960,0.320252,0.363897
13,BE20384,Daily Gut Pulver,2024-04-03,standard,0.0,4004.0,8929,0.000000,0.448426,0.000000,0.624746
58,BE20472,Daily Gut Pulver,2026-03-22,standard,1440.0,2666.0,7744,0.185950,0.344267,0.422783,0.465677
43,BE20439,Daily Collagen Pulver,2025-03-23,standard,753.0,1723.0,5181,0.145339,0.332561,0.427598,0.449752
10,BE20374,Daily Fiber Drink,2024-02-05,standard,510.0,1822.0,5900,0.086441,0.308814,0.498047,0.578046
12,BE20377,Daily Gut Pulver,2024-02-19,standard,425.0,1695.0,5900,0.072034,0.287288,0.460455,0.567079
32,BE20415,Daily Gut Pulver,2024-10-14,standard,267.0,1429.0,5190,0.051445,0.275337,0.356475,0.431071
14,BE20385,Daily Gut Pulver,2024-04-03,standard,0.0,2335.0,8929,0.000000,0.261507,0.000000,0.535796


In [18]:
launch_ratio_table["flag_nc_ratio_too_high"] = (
    launch_ratio_table["nc_ratio_6w_vs_monthly_nc"] > 0.5
)

launch_ratio_table["flag_nc_6w_gt_monthly_nc"] = (
    launch_ratio_table["first_6_week_nc"] > launch_ratio_table["monthly_new_customers_at_launch"]
)

launch_ratio_table["flag_nc_gt_total_customers_1w"] = (
    launch_ratio_table["first_week_nc"] > launch_ratio_table["first_week_total_c"]
)

launch_ratio_table["flag_nc_gt_total_customers_6w"] = (
    launch_ratio_table["first_6_week_nc"] > launch_ratio_table["first_6_week_total_c"]
)

display(
    launch_ratio_table[
        launch_ratio_table[
            [
                "flag_nc_ratio_too_high",
                "flag_nc_6w_gt_monthly_nc",
                "flag_nc_gt_total_customers_1w",
                "flag_nc_gt_total_customers_6w",
            ]
        ].any(axis=1)
    ][
        [
            "sku",
            "product",
            "launch_date",
            "launch_strategy_type",
            "first_week_nc",
            "first_week_total_c",
            "first_6_week_nc",
            "first_6_week_total_c",
            "monthly_new_customers_at_launch",
            "nc_ratio_6w_vs_monthly_nc",
            "nc_share_6w",
            "flag_nc_ratio_too_high",
            "flag_nc_6w_gt_monthly_nc",
            "flag_nc_gt_total_customers_1w",
            "flag_nc_gt_total_customers_6w",
        ]
    ].sort_values("nc_ratio_6w_vs_monthly_nc", ascending=False)
)

,sku,product,launch_date,launch_strategy_type,first_week_nc,first_week_total_c,first_6_week_nc,first_6_week_total_c,monthly_new_customers_at_launch,nc_ratio_6w_vs_monthly_nc,nc_share_6w,flag_nc_ratio_too_high,flag_nc_6w_gt_monthly_nc,flag_nc_gt_total_customers_1w,flag_nc_gt_total_customers_6w
0,BE20351,Gut Restore,2023-12-19,standard,611.0,1352.0,1943.0,3919.0,3037,0.639776,0.495790,True,False,False,False
59,BE20473,Daily Gut Pulver,2026-03-22,standard,2100.0,4873.0,4458.0,8746.0,7744,0.575671,0.509719,True,False,False,False
35,BE20418,Gut Shape,2024-12-27,standard,1069.0,3338.0,2028.0,5573.0,3701,0.547960,0.363897,True,False,False,False


In [19]:
launch_ratio_table["buyer_ratio_1w_existing_clipped"] = (
    launch_ratio_table["buyer_ratio_1w_existing"]
    .clip(
        lower=launch_ratio_table["buyer_ratio_1w_existing"].quantile(0.05),
        upper=launch_ratio_table["buyer_ratio_1w_existing"].quantile(0.95),
    )
)

launch_ratio_table["buyer_ratio_6w_existing_clipped"] = (
    launch_ratio_table["buyer_ratio_6w_existing"]
    .clip(
        lower=launch_ratio_table["buyer_ratio_6w_existing"].quantile(0.05),
        upper=launch_ratio_table["buyer_ratio_6w_existing"].quantile(0.95),
    )
)

launch_ratio_table["nc_ratio_1w_vs_monthly_nc_clipped"] = (
    launch_ratio_table["nc_ratio_1w_vs_monthly_nc"]
    .clip(
        lower=launch_ratio_table["nc_ratio_1w_vs_monthly_nc"].quantile(0.05),
        upper=launch_ratio_table["nc_ratio_1w_vs_monthly_nc"].quantile(0.95),
    )
)

launch_ratio_table["nc_ratio_6w_vs_monthly_nc_clipped"] = (
    launch_ratio_table["nc_ratio_6w_vs_monthly_nc"]
    .clip(
        lower=launch_ratio_table["nc_ratio_6w_vs_monthly_nc"].quantile(0.05),
        upper=launch_ratio_table["nc_ratio_6w_vs_monthly_nc"].quantile(0.95),
    )
)

launch_ratio_table["units_per_customer_1w_clipped"] = (
    launch_ratio_table["units_per_customer_1w"]
    .clip(
        lower=launch_ratio_table["units_per_customer_1w"].quantile(0.05),
        upper=launch_ratio_table["units_per_customer_1w"].quantile(0.95),
    )
)

launch_ratio_table["units_per_customer_6w_clipped"] = (
    launch_ratio_table["units_per_customer_6w"]
    .clip(
        lower=launch_ratio_table["units_per_customer_6w"].quantile(0.05),
        upper=launch_ratio_table["units_per_customer_6w"].quantile(0.95),
    )
)

display(
    launch_ratio_table[
        [
            "sku",
            "product",
            "buyer_ratio_6w_existing",
            "buyer_ratio_6w_existing_clipped",
            "nc_ratio_6w_vs_monthly_nc",
            "nc_ratio_6w_vs_monthly_nc_clipped",
            "units_per_customer_6w",
            "units_per_customer_6w_clipped",
        ]
    ].sort_values("nc_ratio_6w_vs_monthly_nc", ascending=False).head(15)
)

,sku,product,buyer_ratio_6w_existing,buyer_ratio_6w_existing_clipped,nc_ratio_6w_vs_monthly_nc,nc_ratio_6w_vs_monthly_nc_clipped,units_per_customer_6w,units_per_customer_6w_clipped
0,BE20351,Gut Restore,0.035445,0.023808,0.639776,0.443218,1.061240,1.137796
59,BE20473,Daily Gut Pulver,0.026523,0.023808,0.575671,0.443218,1.367139,1.367139
35,BE20418,Gut Shape,0.023358,0.023358,0.547960,0.443218,1.884263,1.581673
13,BE20384,Daily Gut Pulver,0.035048,0.023808,0.448426,0.443218,1.288188,1.288188
58,BE20472,Daily Gut Pulver,0.017361,0.017361,0.344267,0.344267,1.386376,1.386376
43,BE20439,Daily Collagen Pulver,0.015109,0.015109,0.332561,0.332561,1.493083,1.493083
10,BE20374,Daily Fiber Drink,0.018551,0.018551,0.308814,0.308814,1.225254,1.225254
12,BE20377,Daily Gut Pulver,0.017312,0.017312,0.287288,0.287288,1.320174,1.320174
32,BE20415,Daily Gut Pulver,0.014512,0.014512,0.275337,0.275337,1.296531,1.296531
14,BE20385,Daily Gut Pulver,0.023832,0.023808,0.261507,0.261507,1.284305,1.284305


In [20]:
os.makedirs("artifacts", exist_ok=True)

launch_ratio_table.to_csv(
    "artifacts/launch_ratio_table_v2.csv",
    index=False
)

print("Saved: artifacts/launch_ratio_table_v2.csv")

Saved: artifacts/launch_ratio_table_v2.csv


In [21]:
import pickle
import pandas as pd

with open("artifacts/model_artifacts_v2.pkl", "rb") as f:
    artifacts = pickle.load(f)

seg_summary = artifacts["behavioral_segmentation"]["segment_summary"]

display(
    seg_summary.sort_values("avg_monetary", ascending=False)
)

,segment_key,customer_count,avg_recency_days,avg_frequency,avg_monetary,avg_sale_share,global_share
3,SEG_3,16165,269.629261,11.824930,916.680513,0.183545,0.047646
0,SEG_0,147952,395.696516,1.904888,115.301102,0.033639,0.436083
2,SEG_2,42607,323.590771,1.753421,114.515645,0.848412,0.125582
1,SEG_1,132551,1343.813385,1.490724,66.632453,0.001221,0.390689


In [22]:
with open("artifacts/model_artifacts_v2.pkl", "rb") as f:
    artifacts = pickle.load(f)

seg_summary = artifacts["behavioral_segmentation"]["segment_summary"]

display(
    seg_summary[
        [
            "segment_key",
            "segment_label",
            "segment_description",
            "customer_count",
            "global_share",
            "avg_recency_days",
            "avg_frequency",
            "avg_monetary",
            "avg_sale_share",
            "avg_launch_purchase_count_24m",
            "avg_unique_launch_skus_24m",
            "avg_launch_share_24m",
            "avg_unique_product_count_24m",
            "avg_unique_flavour_count_24m",
            "avg_product_diversity_ratio_24m",
        ]
    ].sort_values("avg_monetary", ascending=False)
)

,segment_key,segment_label,segment_description,customer_count,global_share,avg_recency_days,avg_frequency,avg_monetary,avg_sale_share,avg_launch_purchase_count_24m,avg_unique_launch_skus_24m,avg_launch_share_24m,avg_unique_product_count_24m,avg_unique_flavour_count_24m,avg_product_diversity_ratio_24m
3,SEG_3,Loyal high-value buyers,Small or mid-sized segment with high order fre...,4706,0.013871,97.697195,14.445601,1278.604337,0.295575,17.904802,9.508712,0.643654,9.278581,9.226944,0.343138
2,SEG_2,Variety seekers,"Customers who try a wider mix of products, cat...",29834,0.087935,216.964872,6.023463,450.187988,0.247042,5.875947,3.961554,0.627186,4.354528,4.110880,0.486540
1,SEG_1,Launch adopters,Customers with above-average tendency to buy n...,110719,0.326340,340.878178,1.678176,95.326139,0.021404,1.404122,1.257454,0.704508,1.583983,1.307933,0.820929
4,SEG_4,Sale-sensitive buyers,Customers whose purchases are strongly concent...,33723,0.099397,304.273938,1.462503,86.515857,0.879912,1.583430,1.399312,0.689420,1.805000,1.460517,0.813019
0,SEG_0,Dormant low-value customers,"Customers with old last purchase dates, low fr...",160293,0.472457,1246.955725,1.637476,79.033557,0.021522,0.003974,0.003163,0.001477,0.054999,0.006775,0.029288
